## Setup

In [1]:
import json
from datetime import datetime, timedelta
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import Message, ToolParam

client = Anthropic()
model = "claude-haiku-4-5"

## Helper functions

In [2]:
def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })

def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

## Two tools and their schemas

In [3]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(datetime_str, days=0, hours=0, date_format="%Y-%m-%d"):
    dt = datetime.strptime(datetime_str, date_format)
    return (dt + timedelta(days=days, hours=hours)).strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time as a formatted string. Use it whenever the user asks about today's date, the current time, or anything relative to 'now'. The format is set by date_format and defaults to year-month-day hours:minutes:seconds.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {"type": "string", "description": "Python strftime format codes, for example '%Y-%m-%d'. Optional."}
        },
        "required": [],
    },
})

add_duration_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "Adds a number of days and/or hours to a given date and returns the new date as a string. Use it for questions like 'what date is 103 days from today?'. It needs a starting date, normally obtained from get_current_datetime.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {"type": "string", "description": "The starting date, matching date_format, for example '2026-11-01'."},
            "days": {"type": "number", "description": "Days to add. Optional, defaults to 0."},
            "hours": {"type": "number", "description": "Hours to add. Optional, defaults to 0."},
            "date_format": {"type": "string", "description": "Format of the input and output date. Optional, defaults to '%Y-%m-%d'."}
        },
        "required": ["datetime_str"],
    },
})

## Run the tools

In [4]:
def run_tool(name, tool_input):
    if name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    if name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    raise ValueError(f"Unknown tool: {name}")

def run_tools(message):
    results = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        try:
            output = run_tool(block.name, block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": json.dumps(output), "is_error": False})
        except Exception as e:
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": f"Error: {e}", "is_error": True})
    return results

## The conversation loop

In [5]:
def run_conversation(messages):
    tools = [get_current_datetime_schema, add_duration_schema]
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        add_user_message(messages, run_tools(response))
    return messages

The loop stops when Claude is no longer asking for a tool, which ``stop_reason`` tells you.

In [6]:
messages = []
add_user_message(messages, "What day is 103 days from today?")
run_conversation(messages)

for b in messages[-1]["content"]:
    if b.type == "text":
        print(b.text)

103 days from today (November 1, 2026) is **February 12, 2027**.
